# Association rules on the episode transactions

**Pipeline:** `episodes_v1` → transactions → frequent itemsets → **association rules (this notebook)** → cohort analysis.

The frequent itemsets at minsup 200 come from mlxtend `apriori`, which gives the same result as our own implementation (`apriori_from_scratch.ipynb`, section 8). The rules are written from scratch.

**Definitions**, for a frequent itemset $X = A \cup B$ with $A$ and $B$ non-empty and disjoint, and $N$ episodes:

- **support** $\sigma(A \cup B)$: the number of episodes that contain all students of $A$ and $B$;
- **confidence** $c(A \rightarrow B) = \sigma(A \cup B)\,/\,\sigma(A)$: the share of the episodes with $A$ that also contain $B$;
- **lift** $= c(A \rightarrow B)\,/\,(\sigma(B)/N) = N\,\sigma(A \cup B)\,/\,(\sigma(A)\,\sigma(B))$: how many times more often $A$ and $B$ occur together than if they were independent (lift 1).

Confidence ignores how common $B$ is, so it is read together with lift. By downward closure, $\sigma(A)$ and $\sigma(B)$ are always available.

## 1. Frequent itemsets at minsup 200

Load the episodes, build one transaction per episode and find the frequent itemsets. `supports` maps each itemset (a sorted tuple of student ids) to its support count.

In [ ]:
from itertools import combinations
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from mlxtend.frequent_patterns import apriori as mlx_apriori, association_rules as mlx_rules

PROJECT_ROOT = Path.cwd().parent                   # notebooks/ is one level below the project root
PROCESSED = PROJECT_ROOT / "data" / "processed"

sys.path.append(str(PROJECT_ROOT))                 # so "from src..." works inside notebooks/
from src.features.transactions import build_transactions, support_count, to_one_hot

MINSUP = 200                                       # count of episodes, the chosen minimum support

episodes = pd.read_parquet(PROCESSED / "episodes_v1.parquet")
transactions = build_transactions(episodes)        # one transaction per episode
n_episodes = len(transactions)
onehot = to_one_hot(transactions)                  # True/False table for mlxtend

share = (MINSUP - 0.5) / n_episodes                # a fraction just below MINSUP / N, avoids float rounding
frequent = mlx_apriori(onehot, min_support=share, use_colnames=True, low_memory=True)

supports = {}                                      # {sorted tuple of participant ids: support count}
for itemset, support in zip(frequent["itemsets"], frequent["support"]):
    supports[tuple(sorted(int(item) for item in itemset))] = round(support * n_episodes)   # fraction back to a count
print(n_episodes, "episodes,", len(supports), "frequent itemsets at minsup", MINSUP)

## 2. Rules from scratch, checked on a toy example

`make_rules` builds every rule from each frequent itemset; `rules_to_dict` makes two rule tables comparable. On a toy of 5 episodes, the asserts check the number of rules and four values computed by hand.

In [ ]:
def make_rules(supports, n_total):
    """All rules A -> B from the frequent itemsets: A and B are non-empty, disjoint, and A + B is frequent."""
    rows = []
    for itemset, support in supports.items():
        if len(itemset) < 2:                                             # a rule needs at least one student on each side
            continue
        for size in range(1, len(itemset)):                              # size of the antecedent A
            for antecedent in combinations(itemset, size):               # every way to choose A inside the itemset
                consequent = tuple(item for item in itemset if item not in antecedent)  # B is the rest of the itemset
                confidence = support / supports[antecedent]              # sigma(A + B) / sigma(A); A is frequent, so it is in supports
                lift = confidence / (supports[consequent] / n_total)     # confidence divided by how common B is overall
                rows.append({
                    "antecedent": antecedent,
                    "consequent": consequent,
                    "support": support,                                  # episodes that contain A and B
                    "confidence": confidence,
                    "lift": lift,
                })
    return pd.DataFrame(rows)


def rules_to_dict(rules):
    """{(antecedent, consequent): (confidence, lift)}, to compare two rule tables."""
    result = {}
    for row in rules.itertuples():
        result[(row.antecedent, row.consequent)] = (row.confidence, row.lift)
    return result


episode_toy = [{1, 2, 3}, {1, 2, 3, 4}, {1, 2, 3}, {2, 4, 5}, {1, 5, 6}]   # the toy from tests/test_transactions.py
toy_supports = {}                                                          # frequent itemsets at minsup 2, by brute force
for size in range(1, 4):
    for itemset in combinations([1, 2, 3, 4, 5, 6], size):
        count = support_count(episode_toy, itemset)
        if count >= 2:
            toy_supports[itemset] = count
assert len(toy_supports) == 10                                             # the same 10 itemsets as in the Apriori notebook

toy_rules = make_rules(toy_supports, len(episode_toy))
assert len(toy_rules) == 14                                                # 4 pairs x 2 rules + 123 with 3 + 3 splits
toy = rules_to_dict(toy_rules)
assert [round(x, 6) for x in toy[((1,), (2,))]] == [0.75, 0.9375]          # 3/4 and 0.75 / (4/5)
assert [round(x, 6) for x in toy[((1, 2), (3,))]] == [1.0, 1.666667]       # 3/3 and 1 / (3/5)
assert [round(x, 6) for x in toy[((4,), (2,))]] == [1.0, 1.25]             # 2/2 and 1 / (4/5)
assert [round(x, 6) for x in toy[((2,), (4,))]] == [0.5, 1.25]             # 2/4 and 0.5 / (2/5)
toy_rules.round(3)

## 3. All rules, checked against mlxtend

The asserts check that our rules and mlxtend's `association_rules` have the same rules, confidence and lift. The table summarises the rules by the number of students in the antecedent.

In [ ]:
rules = make_rules(supports, n_episodes)
rules["antecedent_size"] = [len(a) for a in rules["antecedent"]]           # students on the left side
rules["consequent_size"] = [len(b) for b in rules["consequent"]]           # students on the right side
mlx = mlx_rules(frequent, num_itemsets=n_episodes, metric="confidence", min_threshold=0.0)   # the library version, every rule

library = {}    # mlx's rules output
for a, b, confidence, lift in zip(mlx["antecedents"], mlx["consequents"], mlx["confidence"], mlx["lift"]):
    key = (tuple(sorted(int(item) for item in a)), tuple(sorted(int(item) for item in b)))   # same format as ours
    library[key] = (confidence, lift)

ours = rules_to_dict(rules)
assert set(ours) == set(library)                                           # the same rules
for key in ours:
    assert abs(ours[key][0] - library[key][0]) < 1e-9                      # the same confidence
    assert abs(ours[key][1] - library[key][1]) < 1e-9                      # the same lift

n_big = sum(1 for itemset in supports if len(itemset) >= 2)                # frequent itemsets that give rules
print(len(rules), "rules from", n_big, "frequent itemsets of size 2 or more, identical to mlxtend")
print("lift > 1:", int((rules["lift"] > 1).sum()), "| confidence >= 0.5:", int((rules["confidence"] >= 0.5).sum()),
      "| confidence >= 0.8:", int((rules["confidence"] >= 0.8).sum()))
display(rules.groupby("antecedent_size").agg(rules=("lift", "size"), median_confidence=("confidence", "median"), median_lift=("lift", "median")).round(3))

## 4. Top rules and the confidence-lift plot

The ten rules with the highest lift and the ten with the highest confidence, then one dot per rule. The grey line is lift 1.

In [ ]:
columns = ["antecedent", "consequent", "support", "confidence", "lift"]
print("Highest lift")
display(rules.sort_values(["lift", "support"], ascending=False).head(10)[columns].round(3).reset_index(drop=True))
print("Highest confidence, ties broken by lift")
display(rules.sort_values(["confidence", "lift"], ascending=False).head(10)[columns].round(3).reset_index(drop=True))

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(rules["confidence"], rules["lift"], s=14, alpha=0.5, color="#2a78d6")
ax.axhline(1, color="#898781", lw=1)                                       # lift 1: as often together as expected by chance
ax.set_yscale("log")                                                       # lift spans orders of magnitude
ax.set_xlabel("confidence")
ax.set_ylabel("lift (log scale)")
ax.set_title(f"Rules at minsup {MINSUP}: confidence against lift")
ax.grid(alpha=0.3)
plt.show()

### Findings (`episodes_v1`, minsup 200)

- The 343 frequent itemsets of two or more students give 2,214 rules, identical to mlxtend's.
- All rules have lift above 1; 1,592 have confidence of at least 0.5 and 720 of at least 0.8.
- As the antecedent grows, the median confidence rises (0.445 for one student, 0.962 for five) and the number of rules falls (938, 775, 400, 95, 6).
- Highest lift: 77.5, for (105, 175, 292) → (243, 309, 427) at support 200. Lift is the same in both directions, confidence is not (0.905 and 0.760). The ten highest-lift rules are re-splits of the same seven students.
- Highest confidence: (321) → (322), 1.0 at support 234: student 321 never appears without student 322.
- Lifts are large because single students appear in few episodes; lift cannot exceed $N/\sigma(B)$.
- **Limit:** lift compares with independence over all episodes and ignores episode size and time of day, so a high lift is not evidence of a social tie. That is the purpose of the null models.